In [51]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

In [81]:
# create geometry mesh
# mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
shape = Rectangle(1,0.5).Face()

shape.edges.Max(X).name="right"
shape.edges.Min(X).name="left"
shape.edges.Max(Y).name="top"
shape.edges.Min(Y).name="bottom"
shape.vertices.Max(X+Y).maxh=0.005
shape.vertices.Min(X-Y).maxh=0.005
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.1))
print(f"n vertices: {mesh.nv}")
print(f"n elements: {mesh.ne}")
Draw(mesh)

n vertices: 117
n elements: 181


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [82]:
# create fe mesh (fe space)
fes = H1(mesh, order=2, dirichlet="bottom|left")
print(f"n total dofs: {fes.ndof}") # number of unknowns in this space
print(f"n free dofs: {fes.FreeDofs()}")

n total dofs: 414
n free dofs: 0: 00100000000001111111111111111111111111111000000000
50: 01111111111111111111111111111111111111111111111111
100: 11111111111111111000111110011011011011011011011011
150: 11111111111111111111111111111111111111111111111111
200: 11111111111111111111111111111111101011011011011010
250: 11101101111111111111111111111111111111111111111111
300: 11111111111111111111111111111111111111111111111111
350: 11111111111111111111111111111111111111111111111111
400: 11111111111111


In [83]:
u = fes.TrialFunction()   # symbolic
v = fes.TestFunction()    # symbolic
gfu = GridFunction(fes)   # solution

In [84]:
# assemble bilinear form
a = BilinearForm(fes)
a += grad(u)*grad(v)*dx
a.Assemble()
print(a.mat)

Row 0:   0: 1.00025   4: -0.488844   50: -0.511411   117: -0.0852351   118: -0.081474   127: 0.166709
Row 1:   1: 0.845439   12: -0.279707   13: -0.297466   59: -0.268266   119: -0.0236915   120: -0.0210195   121: -0.0961954   151: 0.0703094   153: 0.0705971
Row 2:   2: 1.00257   21: -0.465389   22: -0.537185   122: -0.0895308   123: -0.0775649   176: 0.167096
Row 3:   3: 1.0017   40: -0.471687   41: -0.530012   124: -0.0883353   125: -0.0786146   231: 0.16695
Row 4:   0: -0.488844   4: 1.94031   5: -0.282118   50: -0.151788   51: -1.01756   117: 6.6184e-17   118: 0.081474   126: -0.0644914   127: -0.186577   128: -0.0723177   130: 0.111511   259: 0.130401
Row 5:   4: -0.282118   5: 1.75022   6: -0.289733   51: -0.445674   52: -0.732699   126: -0.0350825   128: 0.0821021   129: -0.0600282   130: -0.109108   131: -0.0874854   133: 0.108317   261: 0.101285
Row 6:   5: -0.289733   6: 1.73853   7: -0.326474   52: -0.491182   53: -0.63114   129: -0.0373511   131: 0.0856399   132: -0.0524869

In [85]:
# assemble linear form
f = LinearForm(fes)
f += x*v*dx
f.Assemble()
print(f.vec)

 4.0737e-05
 0.00248535
 5.84138e-06
 7.79857e-09
 0.000433127
 0.000838218
 0.00133978
 0.00192434
 0.00246453
 0.00275967
 0.00289718
 0.00292848
 0.00325476
 0.00347167
 0.00347095
 0.00342133
 0.00182718
 0.000579479
 0.000244508
 0.000116635
 5.484e-05
 3.02052e-05
 3.20931e-05
 7.70341e-05
 0.000167443
 0.000301003
 0.000441505
 0.000905687
 0.0019669
 0.00265027
 0.00264223
 0.00247806
 0.00216346
 0.0015822
 0.000989998
 0.000449519
 0.0001149
 2.85473e-05
 9.79916e-06
 1.73586e-06
 2.1575e-07
 7.11612e-08
 1.27494e-07
 3.36881e-07
 1.34742e-06
 3.83916e-06
 2.88372e-06
 5.98688e-05
 8.9557e-05
 0.000100238
 0.000264722
 0.00124387
 0.00210264
 0.00325141
 0.00471039
 0.00537791
 0.00567284
 0.00564748
 0.005492
 0.00624516
 0.00609327
 0.00716179
 0.00628441
 0.0005741
 0.000463512
 0.000108247
 0.000503675
 0.00084766
 0.00111464
 0.00522529
 0.00512267
 0.00506999
 0.00386754
 0.00258562
 0.00140446
 8.56528e-05
 3.98629e-05
 9.32009e-07
 9.92034e-07
 5.96674e-06
 4.43636e-0

In [86]:
# solve system of equation
print(f"n free dofs: {fes.FreeDofs()}")
gfu.vec.data = a.mat.Inverse(freedofs=fes.FreeDofs()) * f.vec
Draw(gfu)
print(gfu.vec)  # solution

n free dofs: 0: 00100000000001111111111111111111111111111000000000
50: 01111111111111111111111111111111111111111111111111
100: 11111111111111111000111110011011011011011011011011
150: 11111111111111111111111111111111111111111111111111
200: 11111111111111111111111111111111101011011011011010
250: 11101101111111111111111111111111111111111111111111
300: 11111111111111111111111111111111111111111111111111
350: 11111111111111111111111111111111111111111111111111
400: 11111111111111


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

       0
       0
 0.084543
       0
       0
       0
       0
       0
       0
       0
       0
       0
       0
 0.0312321
 0.0545627
 0.0708944
 0.079172
 0.0825709
 0.0838338
 0.0842741
 0.08447
 0.0845311
 0.084537
 0.084497
 0.0843629
 0.0839846
 0.0832893
 0.0820757
 0.0799639
 0.0754402
 0.0681523
 0.059564
 0.0500292
 0.0398188
 0.0291374
 0.0186641
 0.0114895
 0.00747479
 0.00449807
 0.00197313
 0.000664135
       0
       0
       0
       0
       0
       0
       0
       0
       0
       0
 0.00477928
 0.00855391
 0.0129622
 0.0178929
 0.0209005
 0.0224581
 0.0228536
 0.0224459
 0.02523
 0.0482025
 0.065749
 0.0777282
 0.0834069
 0.0841093
 0.0845047
 0.0841274
 0.0835591
 0.0825669
 0.0698938
 0.0618194
 0.0526579
 0.0429034
 0.0327509
 0.0222152
 0.00868041
 0.00486318
 0.000996014
 0.00104747
 0.00176419
 0.00337581
 0.00692094
 0.08439
 0.0843921
 0.0129535
 0.00275382
 0.00756435
 0.00488655
 0.0174536
 0.0120642
 0.0348401
 0.0392463
 0.0412768
 0.0418465
 0.0